In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.dates as mdates

/Users/florentinafabregas/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
# Import data
data_path = '../data/transformed_output.csv'

df = pd.read_csv(data_path)
df.head(2)

,licencePlate,start_time,end_time,lat,lon,parking_time,vehicleTypeId,zipCode,car_type,area_name,day_of_week_start,hour_of_day_start,day_of_week_end,hour_of_day_end
0,bn32098,2025-07-21 17:37:29,2025-07-22 09:13:29,55.658398,12.514628,936,2,2500,car,Valby,Monday,17,Tuesday,9
1,bn32098,2025-07-22 09:17:29,2025-07-22 09:19:29,55.658348,12.515684,2,2,2500,car,Valby,Tuesday,9,Tuesday,9


#### Create movement dataframe

In [9]:
df['start_time'] = pd.to_datetime(df['start_time'])
df['end_time'] = pd.to_datetime(df['end_time'])

df = df.sort_values(by=['licencePlate', 'start_time']).reset_index(drop=True)

# Shift within each licencePlate
df['next_start_time'] = df.groupby('licencePlate')['start_time'].shift(-1)
df['end_lat'] = df.groupby('licencePlate')['lat'].shift(-1)
df['end_lon'] = df.groupby('licencePlate')['lon'].shift(-1)
df['end_zip'] = df.groupby('licencePlate')['zipCode'].shift(-1)
df['end_area_name'] = df.groupby('licencePlate')['area_name'].shift(-1)

movement_df = df.copy()

movement_df['start_move_time'] = movement_df['end_time']
movement_df['end_move_time'] = movement_df['next_start_time']
movement_df['move_duration'] = movement_df['end_move_time'] - movement_df['start_move_time']
movement_df['start_area_name'] = movement_df['area_name']

movement_df = movement_df.dropna(subset=['end_move_time']).copy()

movement_df = movement_df.rename(columns={
    'zipCode': 'start_zip'
})

movement_df['start_lat'] = movement_df['lat']
movement_df['start_lon'] = movement_df['lon']

movement_df = movement_df[[
    'licencePlate',
    'car_type',
    'vehicleTypeId',

    'start_move_time',
    'end_move_time',

    'start_lat', 'start_lon',
    'end_lat', 'end_lon',

    'start_zip', 'end_zip',
    'start_area_name', 'end_area_name',

    #'hour_of_day_end',
    'move_duration'
]]

In [11]:
movement_df.to_csv("../data/movement.csv", index=False)
movement_df.head(3)

,licencePlate,car_type,vehicleTypeId,start_move_time,end_move_time,start_lat,start_lon,end_lat,end_lon,start_zip,end_zip,start_area_name,end_area_name,move_duration
0,bn32098,car,2,2025-07-22 09:13:29,2025-07-22 09:17:29,55.658398,12.514628,55.658348,12.515684,2500,2500.0,Valby,Valby,0 days 00:04:00
1,bn32098,car,2,2025-07-22 09:19:29,2025-07-22 09:23:29,55.658348,12.515684,55.659286,12.519309,2500,1805.0,Valby,Frederiksberg C,0 days 00:04:00
2,bn32098,car,2,2025-07-22 12:10:48,2025-07-22 14:24:49,55.659286,12.519309,55.677685,12.522237,1805,2000.0,Frederiksberg C,Frederiksberg C,0 days 02:14:01
